# 03 - Cargar copias normalizadas al datastore y mosaic dataset

Toma el manifiesto listo de la etapa 02, copia los TIFF normalizados y sus sidecars al datastore, agrega cada raster al mosaic dataset, construye footprints y actualiza campos criticos.

In [ ]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'core').exists() and (candidate / 'flujo_geosupport_final' / 'settings.json').exists():
            return candidate
        if candidate.name.lower() == 'flujo_geosupport_final' and (candidate / 'settings.json').exists():
            parent = candidate.parent
            if (parent / 'core').exists():
                return parent
    raise RuntimeError('No se pudo resolver PROJECT_ROOT: debe existir core/ y flujo_geosupport_final/settings.json')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import flujo_geosupport_final.scripts.geosupport_flujo_completo as flujo
flujo = importlib.reload(flujo)

FINAL_DIR = PROJECT_ROOT / 'flujo_geosupport_final'
SETTINGS_PATH = FINAL_DIR / 'settings.json'
with SETTINGS_PATH.open('r', encoding='utf-8') as file_obj:
    SETTINGS = json.load(file_obj)
flujo.apply_flow_settings(SETTINGS)

## Parametros

In [ ]:
STAGE_02_OUTPUT = FINAL_DIR / 'outputs' / '02_preparar_paths_datastore'
READY_CSV = STAGE_02_OUTPUT / '04_ready_for_datastore.csv'
OUTPUT_DIR = FINAL_DIR / 'outputs' / '03_carga_datastore_mosaico'

# True ejecuta copia, carga al mosaico, BuildFootprints y updates.
# False solo deja los CSV de control sin escribir en datastore/mosaico.
APPLY_CHANGES = bool(SETTINGS.get('apply_changes_default', False))

# True sobrescribe el TIFF y sidecars si ya existen en datastore.
OVERWRITE_DATASTORE_COPY = bool(SETTINGS.get('overwrite_datastore_copy_default', True))

print('Settings:', SETTINGS_PATH)
print('Ready CSV:', READY_CSV)
print('Mosaic dataset:', flujo.PATH_MOSAIC_DATASET)
print('Datastore root:', flujo.DATASTORE_ROOT)
print('Salida:', OUTPUT_DIR)
print('Aplicar cambios:', APPLY_CHANGES)
print('Sobrescribir copia:', OVERWRITE_DATASTORE_COPY)

## Ejecutar carga

In [ ]:
ready_df = pd.read_csv(READY_CSV)

stage = flujo.load_mosaic_stage(
    ready_df=ready_df,
    output_dir=OUTPUT_DIR,
    apply_changes=APPLY_CHANGES,
    overwrite_copy=OVERWRITE_DATASTORE_COPY,
)

display(stage['summary_df'])
display(stage['load_df'].head(20))
display(stage['results_df'].head(20))

## Controles esperados

- `01_load_input_with_attributes.csv`: entrada final con atributos de carga.
- `02_mosaic_load_results.csv`: resultado por imagen.
- `03_errors_review.csv`: registros que requieren revision.
- Las columnas `sidecars_found`, `sidecars_copied` y `sidecars_skipped` permiten validar la copia de `.ovr`, `.aux.xml` y XML asociados.